# 📊 Topic 08 — MLOps Dashboard

Welcome! It's time to bring everything together. 

We have a deployed model, we know how to calculate drift (PSI), and we know how to find anomalies. But we can't run scripts manually every day. We need a **Dashboard**!

## 📡 1. Generating Simulated Production Data

Let's pretend our model has been running in production for 12 weeks. 

We are going to generate some fake data representing how the model has performed over those 12 weeks.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

# 12 weeks of data
weeks = np.arange(1, 13)
week_labels = [f'W{w}' for w in weeks]

# Model Accuracy (Slowly decaying over time)
accuracy = [0.95, 0.94, 0.94, 0.92, 0.90, 0.88, 0.85, 0.82, 0.80, 0.77, 0.75, 0.72]
acc_threshold = 0.80 # If it drops below 80%, we have a problem!

# API Latency in milliseconds (Usually fast, but spiked in week 8)
latency_p95 = [120, 115, 125, 130, 120, 110, 135, 450, 125, 115, 120, 130]
lat_threshold = 200 # If it takes longer than 200ms, the API is too slow!

# Traffic (Number of requests per week)
traffic = [1000, 1100, 1150, 1200, 1300, 1250, 1400, 1500, 1450, 1600, 1700, 1800]

print("✅ Simulated 12 weeks of production data!")

## 🌡️ 2. The Feature Drift Heatmap

Remember the PSI score from Topic 6? Let's pretend we calculated the PSI for 5 different features over the last 12 weeks.

A **Heatmap** is the perfect way to visualize this. Red means high drift!

In [ ]:
# Simulate PSI scores (0.0 to 0.1 is normal, > 0.2 is bad)
# We will make 'income' drift heavily in the later weeks!
features = ['age', 'income', 'credit_score', 'debt_ratio', 'employment_years']
psi_matrix = np.random.uniform(0.01, 0.08, size=(5, 12))
psi_matrix[1, 6:] = np.random.uniform(0.15, 0.35, size=6) # 'income' goes crazy after week 6

plt.figure(figsize=(10, 4))
sns.heatmap(psi_matrix, annot=True, fmt='.2f', xticklabels=week_labels, yticklabels=features, cmap='RdYlGn_r', vmin=0, vmax=0.3)
plt.title('Feature Drift (PSI) Heatmap — Red is Bad!', fontsize=14)
plt.show()

## 🎛️ 3. Building the Master 4-Panel Dashboard

Let's combine Model Accuracy, API Latency, Traffic, and Errors into a single, beautiful dashboard. 

We will also add **Alert Annotations** (red dashed lines) to show when things break our Service Level Objectives (SLOs).

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("MLOps Production Dashboard — Credit Default Model v2.1", fontsize=20, fontweight='bold', y=0.98)

# --- PANEL 1: Model Accuracy ---
axs[0, 0].plot(weeks, accuracy, 'b-o', linewidth=2)
axs[0, 0].axhline(y=acc_threshold, color='red', linestyle='--', label='SLO Alert Threshold (< 80%)')
# Fill the danger zone in red
axs[0, 0].fill_between(weeks, accuracy, acc_threshold, where=[a < acc_threshold for a in accuracy], color='red', alpha=0.3)
axs[0, 0].set_title("Model Accuracy (Decay over time)", fontsize=14)
axs[0, 0].set_xticks(weeks)
axs[0, 0].set_xticklabels(week_labels)
axs[0, 0].legend()

# --- PANEL 2: API Latency ---
axs[0, 1].plot(weeks, latency_p95, 'g-o', linewidth=2)
axs[0, 1].axhline(y=lat_threshold, color='red', linestyle='--', label='Latency Alert (> 200ms)')
axs[0, 1].set_title("API Latency (95th Percentile)", fontsize=14)
axs[0, 1].set_xticks(weeks)
axs[0, 1].set_xticklabels(week_labels)
# Add a text alert for the spike
axs[0, 1].annotate('Server Outage!', xy=(8, 450), xytext=(5, 400), arrowprops=dict(facecolor='black', shrink=0.05), fontsize=12, color='red')
axs[0, 1].legend()

# --- PANEL 3: Traffic Volume ---
axs[1, 0].bar(weeks, traffic, color='#8E44AD', alpha=0.7)
axs[1, 0].set_title("Weekly Prediction Requests (Traffic)", fontsize=14)
axs[1, 0].set_xticks(weeks)
axs[1, 0].set_xticklabels(week_labels)

# --- PANEL 4: Error Rate ---
errors = np.random.uniform(0.1, 0.5, 12)
errors[7] = 8.5 # Huge error spike during the latency outage in week 8
axs[1, 1].plot(weeks, errors, 'r-o', linewidth=2)
axs[1, 1].set_title("API Error Rate (%)", fontsize=14)
axs[1, 1].set_xticks(weeks)
axs[1, 1].set_xticklabels(week_labels)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 💾 4. Exporting the Report

In the real world, you might run a script like this every Monday morning and automatically email the image to your manager.

In [ ]:
import os

# Save the figure we just created!
output_dir = 'reports'
os.makedirs(output_dir, exist_ok=True)
filepath = f"{output_dir}/weekly_mlops_dashboard.png"

fig.savefig(filepath, dpi=150, bbox_inches='tight')
print(f"\u2705 Dashboard successfully saved as an image at: {filepath}")

## 📝 Key Takeaways & Quiz

### Key Takeaways:
- **Combine your metrics:** A drop in accuracy might be caused by Data Drift (look at the heatmap!). An API error spike might be caused by Latency issues. Dashboards let you see the connections.
- **Alerts need action:** Don't set an alert if you don't plan to do anything when it goes off.
- **Google's Golden Signals:** Always track Latency, Traffic, Errors, and Saturation.

### ❓ Quiz (Check your understanding)
1. Look at the dashboard we built. In which week did the model accuracy finally drop below the acceptable SLO threshold?
2. Look at the Heatmap. Which feature caused the model to start decaying?
3. Why did we use `fill_between` in red on the Model Accuracy chart?